# Planner Node — Unit Tests

Two passes (legacy polish + new ops from leftover rules) plus a gap check.

**Pre-requisites:** files under `data/inputs/`, `OPENAI_API_KEY` in `.env`,
Qdrant up (Planner degrades gracefully if a collection is missing).

In [1]:
# Step 1 — Imports and path setup
#
# Test fixtures (which spec, rules, legacy to use) live in
# openapi_generator.config.paths, so the notebook stays free of glob magic
# and ad-hoc strings. Edit paths.py to switch fixtures.

import json
from pathlib import Path

import yaml

from openapi_generator.config import get_logger
from openapi_generator.config.paths import (
    TEST_LEGACY_PATH,
    TEST_RULES_PATH,
    TEST_SPEC_PATH,
)
from openapi_generator.nodes.loader import loader_node
from openapi_generator.nodes.planner import planner_node

logger = get_logger(__name__)

SPEC_PATH   = Path(TEST_SPEC_PATH).resolve()
RULES_PATH  = Path(TEST_RULES_PATH).resolve()
LEGACY_PATH = Path(TEST_LEGACY_PATH).resolve()

logger.info(f"Spec   : {SPEC_PATH}")
logger.info(f"Rules  : {RULES_PATH}")
logger.info(f"Legacy : {LEGACY_PATH if LEGACY_PATH.is_file() else '(not present)'}")
assert SPEC_PATH.is_file()
assert RULES_PATH.is_file()

/home/arimatea/Documents/Pessoal/Mestrado/0-Mestrado_Unicamp_2025/5-Projeto_mestrado_ericsson/openapi_multiagents/workspace/openapi_generator/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-05-26 22:58:46 [INFO] __main__: Spec   : /home/arimatea/Documents/Pessoal/Mestrado/0-Mestrado_Unicamp_2025/5-Projeto_mestrado_ericsson/openapi_multiagents/workspace/openapi_generator/data/inputs/3gpp/28532-i00.md
2026-05-26 22:58:46 [INFO] __main__: Rules  : /home/arimatea/Documents/Pessoal/Mestrado/0-Mestrado_Unicamp_2025/5-Projeto_mestrado_ericsson/openapi_multiagents/workspace/openapi_generator/data/inputs/rules/rules_bank_28532-i00_full_20260427_214528.json
2026-05-26 22:58:46 [INFO] __main__: Legacy : /home/arimatea/Documents/Pessoal/Mestrado/0-Mestrado_Unicamp_2025/5-Projeto_mestrado_ericsson/openapi

In [2]:
# Step 2 — Pre-parse rules_bank and legacy. Same trick as the loader notebook.

with open(RULES_PATH, "r", encoding="utf-8") as f:
    rules_bank = json.load(f)

legacy_openapi = None
if LEGACY_PATH.is_file():
    with open(LEGACY_PATH, "r", encoding="utf-8") as f:
        legacy_openapi = yaml.safe_load(f)

rules = rules_bank.get("rules", [])
logger.info(f"rules    : {len(rules)}")
logger.info(f"legacy   : {'present' if legacy_openapi else 'absent'}")
if legacy_openapi:
    logger.info(f"  paths   : {len(legacy_openapi.get('paths') or {})}")

2026-05-26 22:58:46 [INFO] __main__: rules    : 240
2026-05-26 22:58:46 [INFO] __main__: legacy   : present
2026-05-26 22:58:46 [INFO] __main__:   paths   : 1


In [3]:
# Step 3 — Run Loader first so the state is realistic (Planner is fed the
# Loader output, not raw inputs).

loader_state = {
    "spec_doc_path": str(SPEC_PATH),
    "rules_bank": rules_bank,
    "legacy_openapi": legacy_openapi,
}
loader_out = loader_node(loader_state)
logger.info(f"Loader produced {len(loader_out['parsed_spec_sections'])} sections")

# Compose the state for the Planner the way LangGraph would: Loader writes
# merged onto the initial state.
state_A = {**loader_state, **loader_out}

2026-05-26 22:58:46 [INFO] openapi_generator.nodes.loader: Loader → parsed 696 sections from 28532-i00.md (excluded 1 symbolic-title section(s))
2026-05-26 22:58:46 [INFO] openapi_generator.nodes.loader: Loader → rules_bank: 240 rule(s); legacy_openapi: present
2026-05-26 22:58:46 [INFO] openapi_generator.nodes.loader: Loader → seeding final_openapi from legacy (1 path(s), 16 schema(s))
2026-05-26 22:58:46 [INFO] __main__: Loader produced 696 sections


In [4]:
# Step 4 — Run the Planner

out = planner_node(state_A)
ops = out["operations_plan"]
gaps = (out.get("extraction_plan") or {}).get("probable_gaps") or []

logger.info(f"Operations : {len(ops)}")
for op in ops:
    logger.info(
        f"  {op['action']:>7} {op['method'].upper():>6} {op['path']}  "
        f"(rules={len(op['source_rule_ids'])})"
    )
logger.info(f"Probable gaps: {len(gaps)}")

2026-05-26 22:58:46 [INFO] openapi_generator.nodes.planner: Planner Node started (2 passes + gap check).
2026-05-26 22:58:47 [INFO] openapi_generator.config.llm_config: Default LLM ready: model=gpt-4.1-mini temperature=0.0
2026-05-26 22:58:47 [INFO] openapi_generator.nodes.planner: Planner Pass 1 → 4 legacy operation(s) to review.
2026-05-26 22:58:48 [INFO] openapi_generator.rag.qdrant_factory: Connecting to Qdrant at localhost:6333
2026-05-26 22:58:48 [INFO] openapi_generator.config.hardware: Embedding device: cuda
2026-05-26 22:58:48 [INFO] openapi_generator.rag.qdrant_factory: Loading embeddings: sentence-transformers/all-MiniLM-L6-v2 on cuda
/home/arimatea/Documents/Pessoal/Mestrado/0-Mestrado_Unicamp_2025/5-Projeto_mestrado_ericsson/openapi_multiagents/workspace/openapi_generator/.venv/lib/python3.13/site-packages/qdrant_client/qdrant_remote.py:282: UserWarning: Qdrant client version 1.18.0 is incompatible with server version 1.16.3. Major versions should match and minor version d

In [5]:
# Step 5 — Persist the ExtractionPlan to data/outputs/test_planner/

from datetime import datetime
from openapi_generator.config.paths import OUTPUTS_TEST_PLANNER_DIR

OUT_DIR = Path(OUTPUTS_TEST_PLANNER_DIR).resolve()
OUT_DIR.mkdir(parents=True, exist_ok=True)
ts = datetime.now().strftime("%Y%m%d_%H%M%S")
plan_path = OUT_DIR / f"plan_{ts}.json"
plan_path.write_text(
    json.dumps(out.get("extraction_plan") or {}, indent=2, ensure_ascii=False),
    encoding="utf-8",
)
logger.info(f"Wrote ExtractionPlan: {plan_path}")

2026-05-26 23:03:48 [INFO] __main__: Wrote ExtractionPlan: /home/arimatea/Documents/Pessoal/Mestrado/0-Mestrado_Unicamp_2025/5-Projeto_mestrado_ericsson/openapi_multiagents/workspace/openapi_generator/data/outputs/test_planner/plan_20260526_230348.json
